In [4]:
# this script takes the file until 2025 Q3 and matches the columns song, album, country, platform


# Import main data file
import pandas as pd
import os
from opencc import OpenCC


file ='Earth_2022Q4_2025Q4_1_raw_combined.csv'

inputdirectory = '../../50 KM Group/Royalties/Statements/Karen/Earth/Combined Statements/'
outputdirectory = '../../50 KM Group/Royalties/Statements/Karen/_output/'
outputfilename1 = 'Earth_2022Q4_2025Q4_2_matched.csv'
outputfilename2 = 'Earth_2022Q4_2025Q4_2_matched_key_columns.csv'
globalinputdirectory = '../../50 KM Group/Royalties/Statements/Karen/All labels combined/lookup_tables/'

lookup_country = 'lookup_tables/Earth_Lookup_Country.csv'
lookup_platform = 'lookup_tables/Earth_Lookup_platform.csv'
lookup_isrc_song_album_albumtype = 'lookup_tables/Earth_Lookup_isrc_song_album_albumtype.xlsx'

converter = OpenCC('s2t') 

def readfile(directory,file):
    path = os.path.join(directory, file)
    df = pd.read_csv(path,low_memory=False)
    print(f"The dataframe of the file '{file}' has {df.shape[0]} rows and {len(df.columns)} columns.")
    return df

def readfilexls(directory,file,sheet):
    path = os.path.join(directory, file)
    df = pd.read_excel(path,sheet_name=sheet)
    print(f"The dataframe of the file '{file}' has {df.shape[0]} rows and {len(df.columns)} columns.")
    return df

data = readfile(inputdirectory,file)
print(f"Total fee: {data['Royalties (USD)'].sum()}. Total units: {data['Units'].sum()}")
df_country = readfile(inputdirectory,lookup_country)
df_platform = readfile(inputdirectory,lookup_platform)
df_isrc_song_album_albumtype= readfilexls(inputdirectory,lookup_isrc_song_album_albumtype,"Data")


The dataframe of the file 'Earth_2022Q4_2025Q4_1_raw_combined.csv' has 1556703 rows and 63 columns.
Total fee: 610175.0151680243. Total units: 354741920.0
The dataframe of the file 'lookup_tables/Earth_Lookup_Country.csv' has 458 rows and 2 columns.
The dataframe of the file 'lookup_tables/Earth_Lookup_platform.csv' has 94 rows and 2 columns.
The dataframe of the file 'lookup_tables/Earth_Lookup_isrc_song_album_albumtype.xlsx' has 902 rows and 5 columns.


In [5]:

# Characterise the DAta frame
empty_rows_before_album_song = data[data['ISRC'].isna() &data['Album'].isna() & data['Song'].isna()]
count_of_rows = len(empty_rows_before_album_song)
print(f"\nThere are a total of {count_of_rows} Rows that have no entry in ISRC, 'Album' and 'Song'")
data.fillna({'ISRC': 'XX_UNKNOWN'}, inplace=True)

def merge(df1, df2, col):
    print(f"\nMerging on {col}:")
    empty_cells = df1[col].isna().sum()
    print(f"There are a total of {empty_cells} rows that have no entry in {col}")
    print(f"Total fee: {df1['Royalties (USD)'].sum()}. Total units: {df1['Units'].sum()}")
    df1.fillna({col: 'XX_UNKNOWN'}, inplace=True)
    empty_cells = 0
    df_merged = pd.merge(df1, df2, on=col, how='left')
    print(f"Total fee: {df_merged['Royalties (USD)'].sum()}. Total units: {df_merged['Units'].sum()}")
    new_columns = df_merged.columns.difference(df1.columns)
    first_new_col = new_columns[0] if not new_columns.empty else None
    empty_cells2 = df_merged[first_new_col].isna().sum() if first_new_col else 0
    diff_empty = empty_cells2 - empty_cells
    if diff_empty == 0:
        print(f"Merging of column {col} was successful")
    else:
        print(f"Merging with issues. There are a total of {diff_empty} cells that could not be matched (see 'match_issues_{col}.xlsx').")
        empty_rows = df_merged[df_merged[first_new_col].isna()].copy()
        empty_rows["Row_Number"] = empty_rows.index 
        col=col.replace('/','')
        empty_rows.to_excel(f"{outputdirectory}/match_issues_{col}.xlsx", engine='openpyxl', index=False)
    print(f"The new dataframe has {df_merged.shape[0]} rows and {len(df_merged.columns)} columns.")
    return df_merged 

def mergealbum(df1, df2, col):
    print(f"\nMerging on {col}:")
    df_merged = pd.merge(df1, df2, on=col, how='left')
    new_columns = df_merged.columns.difference(df1.columns)
    first_new_col = new_columns[0] if not new_columns.empty else None
    empty_cells2 = df_merged[first_new_col].isna().sum() if first_new_col else 0
    if empty_cells2 == 0:
        print(f"Merging of column {col} was successful")
    else:
        print(f"Merging with issues. There are a total of {empty_cells2} cells that could not be matched (see 'match_issues_album_{col}.xlsx').")
        empty_rows = df_merged[df_merged[first_new_col].isna()]
        empty_rows.to_excel(f"{outputdirectory}/match_issues_album_{col}.xlsx", engine='openpyxl', index=False)
    print(f"The new dataframe has {df_merged.shape[0]} rows and {len(df_merged.columns)} columns.")
    return df_merged 


df_merged = merge(data,df_country,'Country')
df_merged = merge(df_merged,df_platform,'Platform')

df_merged['Song'] = df_merged['Song'].astype(str)
df_merged['Album'] = df_merged['Album'].astype(str)

df_merged['ISRC.Song.Album'] = 'mod.' + df_merged['ISRC'] + '.' + df_merged['Song'] + '.' + df_merged['Album']

df_merged['ISRC.Song.Album'] = df_merged['ISRC.Song.Album'].apply(converter.convert)

df_merged = df_merged.rename(columns={'ISRC': 'ISRC_original'})
df_merged = df_merged.rename(columns={'Song': 'Song_original'})
df_merged = df_merged.rename(columns={'Album': 'Album_original'})

df_merged['ISRC.Song.Album_MOD'] = (df_merged['ISRC.Song.Album']
            .str.replace(" ", '', regex=False)
            .str.replace("(", '', regex=False)
            .str.replace(")", '', regex=False)
            .str.replace("-", '', regex=False)
            .str.replace("/", '', regex=False)
            .str.replace('（', '', regex=False)
            .str.replace('）', '', regex=False)
            .str.replace('’', '', regex=False)
            .str.replace("'", '', regex=False)
            .str.lower())

df_merged = merge(df_merged,df_isrc_song_album_albumtype,'ISRC.Song.Album_MOD')
unused_rows = df_isrc_song_album_albumtype[~df_isrc_song_album_albumtype['ISRC.Song.Album_MOD'].isin(df_merged['ISRC.Song.Album_MOD'])]
output_path = os.path.join(outputdirectory, 'unused_lookup_rows_ISRC_Song_Album.csv')
unused_rows.to_csv(output_path, index=False)  

royalties = df_merged['Royalties (USD)'].sum()
units = df_merged['Units'].sum()
print(f"Total fee: {royalties}. Total units: {units}")

# drop not needed 2 columns and rename others
print("\nDrop not needed columns:")
df_merged = df_merged.drop(columns=['ISRC.Song.Album'])
df_merged = df_merged.drop(columns=['ISRC.Song.Album_MOD'])
df_merged = df_merged.drop(columns=['备注'])
df_merged = df_merged.loc[:, ~df_merged.columns.str.startswith('Unnamed')]
df_merged = df_merged.rename(columns={'Platform': 'Platform orig'})
df_merged = df_merged.rename(columns={'Platform new': 'Platform'})
df_merged = df_merged.rename(columns={'Country': 'Country orig'})
df_merged = df_merged.rename(columns={'Country new': 'Country'})

df_merged.fillna({'Statement Quarter':'XX_UNKNOWN',
                  'Sales Quarter':'XX_UNKNOWN',
                  'Sales Month':'XX_UNKNOWN',
                  'Album':'XX_UNKNOWN',
                  'Country':'XX_UNKNOWN',
                  'Platform':'XX_UNKNOWN',
                  'Song':'XX_UNKNOWN',
                  'Album Type':'XX_UNKNOWN'}, inplace=True)
df_merged[['Units','Royalties (USD)','Royalties (CNY)']]=df_merged[['Units','Royalties (USD)','Royalties (CNY)']].fillna(0)
print(f"The final dataframe has {df_merged.shape[0]} rows and {len(df_merged.columns)} columns.")

def final_review(col):
    count = df_merged[col].str.contains('XX_UNKNOWN').sum()
    print(f"Rows with no entry for {col}: {count}")

df_merged = df_merged.sort_index(axis=1)

print(f"The final dataframe has {df_merged.shape[0]} rows and {len(df_merged.columns)} columns.")


output_path = os.path.join(outputdirectory, outputfilename1)
df_merged.to_csv(output_path, index=False)

for column in df_merged.columns:        
        print(column)
        
final_review('Country')
final_review('Platform')
final_review('Album')
final_review('Song')
final_review('Album Type')

subset_df = df_merged[[
    'Statement Quarter',
    'Sales Quarter',
    'Sales Month',
    'Country',
    'Platform',
    'ISRC',
    'Album',
    'Units',
    'Share MABB (CNY)',
    'Royalties (CNY)'
]]

output_path = os.path.join(outputdirectory, outputfilename2)
subset_df.to_csv(output_path, index=False)








There are a total of 3 Rows that have no entry in ISRC, 'Album' and 'Song'

Merging on Country:
There are a total of 877 rows that have no entry in Country
Total fee: 610175.0151680243. Total units: 354741920.0
Total fee: 610175.0151680243. Total units: 354741920.0
Merging of column Country was successful
The new dataframe has 1556703 rows and 64 columns.

Merging on Platform:
There are a total of 0 rows that have no entry in Platform
Total fee: 610175.0151680243. Total units: 354741920.0
Total fee: 610175.0151680243. Total units: 354741920.0
Merging of column Platform was successful
The new dataframe has 1556703 rows and 65 columns.

Merging on ISRC.Song.Album_MOD:
There are a total of 0 rows that have no entry in ISRC.Song.Album_MOD
Total fee: 610175.0151680243. Total units: 354741920.0
Total fee: 610175.0151680243. Total units: 354741920.0
Merging of column ISRC.Song.Album_MOD was successful
The new dataframe has 1556703 rows and 71 columns.
Total fee: 610175.0151680243. Total unit

In [6]:
# for col in df_merged.columns:
#     print(f"Column: {col}")
#     print(df_merged[col].map(type).value_counts())
#     print()